# Credit Risk Detection — Full MLOps Pipeline

**University MLOps Group 10**

This notebook runs the complete credit risk detection pipeline end-to-end 


> **Note**: The data pipeline (PDF extraction, label building, embedding generation) has already been run and the outputs are committed to the repo under `data/processed/`, `data/embeddings/`, `data/labels/`, and `data/finetune/`. This notebook skips those steps and goes straight to inference.

---
## Section 1: Setup

Check the GPU, clone the repository, install all Python dependencies, and configure environment variables.


In [ ]:
# Verify GPU availability — you should see a T4 (or better)
!nvidia-smi

In [ ]:
import os


GITHUB_REPO_URL = "https://github.com/pankti0/MLOps-group-10"  
USE_MANUAL_UPLOAD = False  # Set True to skip cloning and upload manually

REPO_DIR = "/content/MLOps-group-10"

if USE_MANUAL_UPLOAD:
    print("Manual upload mode selected.")
    print("Please upload the repo zip via Files panel and unzip to /content/MLOps-group-10")
    print("  !unzip /content/MLOps-group-10.zip -d /content/")
elif not os.path.isdir(REPO_DIR):
    print(f"Cloning {GITHUB_REPO_URL} ...")
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
    print("Clone complete.")
else:
    print(f"Repo already exists at {REPO_DIR}, pulling latest...")
    !git -C {REPO_DIR} pull

In [ ]:
# Install all project dependencies.
print("Installing dependencies from requirements.txt ...")
!pip install -q -r /content/MLOps-group-10/requirements.txt

# Ensure bitsandbytes is the CUDA-enabled version 
!pip install -q bitsandbytes>=0.41.0

print("All dependencies installed.")

In [ ]:

# Configure environment variables.
#


try:
    from google.colab import userdata
    # Try loading from Colab Secrets
    HF_TOKEN = userdata.get('HF_TOKEN')
    WANDB_API_KEY = userdata.get('WANDB_API_KEY')
    print("Loaded secrets from Colab Secrets vault.")
except Exception:
    # Fallback: assign directly
    HF_TOKEN = "hf_YOUR_TOKEN_HERE"          
    WANDB_API_KEY = "YOUR_WANDB_API_KEY_HERE" 
    print("Using directly-assigned API keys (Colab Secrets not available).")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN  
os.environ["WANDB_API_KEY"] = WANDB_API_KEY


env_content = f"""HF_TOKEN={HF_TOKEN}
HUGGING_FACE_HUB_TOKEN={HF_TOKEN}
WANDB_API_KEY={WANDB_API_KEY}
"""
with open("/content/MLOps-group-10/.env", "w") as f:
    f.write(env_content)

print("Environment variables configured.")

In [ ]:
# Run RAG inference for fold 1 test split
!python3 scripts/run_inference_all.py --approach rag --fold 1 --split test

---
## Section 2: Weights & Biases Login

All training runs, inference runs, and evaluation metrics are logged to W&B for experiment tracking.
If you haven't already, create a free account at [wandb.ai](https://wandb.ai).

In [ ]:
import wandb

# Log in using the API key set in Section 1.
# If WANDB_API_KEY env var is set, this proceeds non-interactively.
wandb.login()
print(f"W&B version: {wandb.__version__}")
print("Logged in to Weights & Biases successfully.")

---
## Section 3: Verify Pre-Processed Data

The following data artefacts were generated by the data pipeline and are committed to the repo.
This cell verifies they all exist before running inference:

- `data/processed/` — per-company JSON files with extracted 10-K sections
- `data/labels/company_labels.csv` — ground-truth credit risk labels
- `data/embeddings/faiss.index` + `chunk_metadata.json` — vector store for RAG
- `data/finetune/train.jsonl`, `val.jsonl`, `test.jsonl` — fine-tuning data

In [ ]:
import os
import pandas as pd

REPO_ROOT = "/content/MLOps-group-10"


required_paths = {
    "Labels CSV"       : f"{REPO_ROOT}/data/labels/company_labels.csv",
    "FAISS index"      : f"{REPO_ROOT}/data/embeddings/faiss.index",
    "Chunk metadata"   : f"{REPO_ROOT}/data/embeddings/chunk_metadata.json",
    "Train JSONL"      : f"{REPO_ROOT}/data/finetune/train.jsonl",
    "Val JSONL"        : f"{REPO_ROOT}/data/finetune/val.jsonl",
    "Test JSONL"       : f"{REPO_ROOT}/data/finetune/test.jsonl",
}

all_ok = True
for name, path in required_paths.items():
    exists = os.path.isfile(path)
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {name}: {path}")
    if not exists:
        all_ok = False


processed_files = [f for f in os.listdir(f"{REPO_ROOT}/data/processed/") if f.endswith(".json") and "sections" in f]
print(f"\n  [OK] Processed sections: {len(processed_files)} companies")
for f in sorted(processed_files):
    print(f"       {f}")

if all_ok:
    print("\nAll required data files present — ready to run inference.")
else:
    print("\nWARNING: Some files are missing. Re-run the data pipeline scripts if needed.")

In [ ]:
# Display company labels — these are the ground-truth credit risk labels
import pandas as pd
from IPython.display import display

labels_df = pd.read_csv("/content/MLOps-group-10/data/labels/company_labels.csv")

def style_risk(val):
    colors = {"low": "#d4edda", "medium": "#fff3cd", "high": "#f8d7da"}
    return f"background-color: {colors.get(str(val).lower(), '')}"

print(f"Dataset: {len(labels_df)} companies")
display(
    labels_df.style
    .applymap(style_risk, subset=["risk_category"])
    .format({"risk_score": "{:.0f}"})
    .set_caption("Company Credit Risk Labels (Ground Truth)")
)

In [ ]:
# Show FAISS index statistics
import json
import faiss

index = faiss.read_index("/content/MLOps-group-10/data/embeddings/faiss.index")
with open("/content/MLOps-group-10/data/embeddings/chunk_metadata.json") as f:
    metadata = json.load(f)

print("FAISS Index Statistics")
print(f"  Total vectors : {index.ntotal}")
print(f"  Dimension     : {index.d}")
print(f"  Metadata chunks: {len(metadata)}")

# Show a sample chunk
if metadata:
    sample = metadata[0]
    print(f"\nSample chunk keys: {list(sample.keys())}")
    print(f"  ticker : {sample.get('ticker', 'N/A')}")
    print(f"  section: {sample.get('section', 'N/A')}")
    print(f"  text   : {str(sample.get('text', ''))[:120]}...")

---
## Section 4: Baseline Inference

Run credit risk inference using **direct prompting** (no retrieval, no fine-tuning).

- Model: `mistralai/Mistral-7B-Instruct-v0.2`
- Quantization: 4-bit NF4 via `bitsandbytes` 
- The model reads raw 10-K section text and outputs a risk score + label



## ⚡️ Standardized Stratified K-Fold Cross-Validation (NEW)

All experiments now use **stratified k-fold cross-validation** for robust, fair, and reproducible evaluation.

- Use the `--fold` and `--split` arguments in all inference commands.

- Results are saved as `results/{approach}_fold{N}_{split}.csv`.

- Update all result display code to use the correct fold/split files.

You can loop over all folds for full evaluation.

In [ ]:
# Example: Run baseline inference for fold 1, test split
!python3 scripts/run_inference_all.py --approach baseline --fold 1 --split test

In [ ]:
import pandas as pd
import os

# Show available fine-tune splits for fold 1
for split in ["train", "val", "test"]:
    path = f"/content/MLOps-group-10/data/finetune/fold_1/{split}.jsonl"
    if not os.path.exists(path):
        print(f"[WARN] {split} split not found: {path}")
        continue
    print(f"\n=== {split.upper()} split (fold 1) ===")
    with open(path) as f:
        for i, line in enumerate(f):
            if i >= 3:
                break
            print(line.strip())
    # Optionally, show count
    with open(path) as f:
        count = sum(1 for _ in f)
    print(f"Total examples: {count}")

---
## Section 5: RAG Inference

Run credit risk inference using **Retrieval-Augmented Generation (RAG)**.

- The FAISS index (built from chunked 10-K text) is used to retrieve the most relevant passages for each company
- Retrieved context is prepended to the prompt before calling Mistral-7B
- This typically improves grounding compared to baseline (less hallucination)

The same quantized model from Section 4 is reused (Colab caches the weights).

In [ ]:
# Example: Run RAG inference for fold 1, test split
!python3 scripts/run_inference_all.py --approach rag --fold 1 --split test

In [ ]:
# Display RAG predictions
import pandas as pd
from IPython.display import display

rag_path = "/content/MLOps-group-10/data/results/rag_predictions.csv"

if os.path.isfile(rag_path):
    rag_df = pd.read_csv(rag_path)
    display_cols = ["ticker", "company_name", "predicted_score", "predicted_label", "risk_level"]

    print(f"RAG predictions: {len(rag_df)} companies")
    display(
        rag_df[display_cols].style
        .applymap(style_risk_level, subset=["risk_level"])
        .format({"predicted_score": "{:.1f}"})
        .set_caption("RAG Inference Results")
    )

    # Quick comparison: baseline vs RAG scores
    if os.path.isfile(baseline_path):
        baseline_df = pd.read_csv(baseline_path)
        compare = baseline_df[["ticker", "predicted_score"]].rename(columns={"predicted_score": "baseline_score"})
        compare = compare.merge(rag_df[["ticker", "predicted_score"]].rename(columns={"predicted_score": "rag_score"}), on="ticker")
        compare["delta"] = compare["rag_score"] - compare["baseline_score"]
        print("\nBaseline vs RAG score comparison:")
        display(compare.style.format("{:.1f}", subset=["baseline_score", "rag_score", "delta"]))
else:
    print(f"Predictions file not found at {rag_path}. Check inference output above for errors.")

---
## Section 6: Generate Fine-Tune Data 

The fine-tuning data (`train.jsonl`, `val.jsonl`, `test.jsonl`) is already committed to the repo and should be present.

This cell re-generates it only if the files are missing or you want a fresh split.
The script creates instruction-tuning examples in the format `{instruction, input, output}` from the 10-K sections and labels.

In [ ]:
import os

train_path = "/content/MLOps-group-10/data/finetune/train.jsonl"
val_path   = "/content/MLOps-group-10/data/finetune/val.jsonl"
test_path  = "/content/MLOps-group-10/data/finetune/test.jsonl"

FORCE_REGENERATE = False  # Set True to regenerate even if files exist

if not all(os.path.isfile(p) for p in [train_path, val_path, test_path]) or FORCE_REGENERATE:
    print("Generating fine-tune data...")
    !python3 scripts/generate_finetune_data.py
else:
    print("Fine-tune data already present — skipping generation.")

# Show example counts
for split, path in [("train", train_path), ("val", val_path), ("test", test_path)]:
    if os.path.isfile(path):
        with open(path) as f:
            n = sum(1 for line in f if line.strip())
        print(f"  {split:5s}: {n} examples  ({path})")
    else:
        print(f"  {split:5s}: FILE MISSING")

In [ ]:
# Show a sample training example
import json

with open("/content/MLOps-group-10/data/finetune/train.jsonl") as f:
    first_example = json.loads(f.readline())

print("Sample training example:")
print(json.dumps(first_example, indent=2)[:1000], "..." if len(json.dumps(first_example)) > 1000 else "")

---
## Section 7: LoRA Fine-Tuning (Extra Mile — 3 configs)

Fine-tune Mistral-7B using **LoRA (Low-Rank Adaptation)** with 4-bit quantization via QLoRA.

Three configurations are trained to compare rank vs. performance vs. compute cost:

| Config | Rank `r` | Alpha | Target modules | W&B run |
|--------|----------|-------|---------------|---------|
| `lora_config_r8.yaml` | 8 | 16 | q, v only | `lora-r8` |
| `lora_config.yaml` | 16 | 32 | q, k, v, o | `lora-sft` |
| `lora_config_r32.yaml` | 32 | 64 | q, k, v, o, up, down | `lora-r32` |

All runs use: 3 epochs, batch size 2, grad accum 4, lr 2e-4, cosine scheduler, bf16.

> **Already ran r=16?** Skip the first cell below and only run r=32 and r=8.

In [ ]:
# Train LoRA for fold 1
!python3 scripts/train_lora.py --fold 1

In [ ]:
!python scripts/train_lora.py --config configs/lora_config_r32.yaml

In [ ]:
!python scripts/train_lora.py --config configs/lora_config_r8.yaml

In [ ]:
# Verify the adapter was saved
import os

adapter_dir = "/content/MLOps-group-10/data/models/lora_adapter"
final_adapter_dir = os.path.join(adapter_dir, "final_adapter")

for d in [adapter_dir, final_adapter_dir]:
    if os.path.isdir(d):
        files = os.listdir(d)
        print(f"  {d}")
        for f in sorted(files):
            size_mb = os.path.getsize(os.path.join(d, f)) / 1e6
            print(f"    {f}  ({size_mb:.1f} MB)")
    else:
        print(f"  NOT FOUND: {d}")

# Show training loss from W&B logs if available
print("\nTraining logs are available in your W&B dashboard:")
print("  https://wandb.ai/home → project: credit-risk-detection → run: lora-sft")

In [ ]:
# OptionalPull training loss curve from W&B and plot locally
try:
    import wandb
    import matplotlib.pyplot as plt

    api = wandb.Api()
    # Find the most recent lora-sft run
    runs = api.runs("credit-risk-detection", filters={"display_name": "lora-sft"})
    runs = sorted(runs, key=lambda r: r.created_at, reverse=True)

    if runs:
        run = runs[0]
        history = run.history(keys=["train/loss", "eval/loss"], pandas=True)

        if not history.empty:
            fig, ax = plt.subplots(figsize=(8, 4))
            if "train/loss" in history.columns:
                ax.plot(history["train/loss"].dropna(), label="Train loss")
            if "eval/loss" in history.columns:
                ax.plot(history["eval/loss"].dropna(), label="Val loss")
            ax.set_xlabel("Step")
            ax.set_ylabel("Loss")
            ax.set_title("LoRA Fine-Tuning Loss Curve")
            ax.legend()
            plt.tight_layout()
            plt.savefig("/content/MLOps-group-10/data/results/lora_loss_curve.png", dpi=150)
            plt.show()
            print(f"Loss curve saved. Run: {run.name} | URL: {run.url}")
        else:
            print("No loss history found in W&B run (may still be syncing).")
    else:
        print("No W&B runs found for lora-sft. Check your W&B dashboard manually.")
except Exception as e:
    print(f"Could not fetch W&B loss curve: {e}")
    print("Check your W&B dashboard at https://wandb.ai")

---
## Section 8: LoRA Inference

Run inference using the **LoRA fine-tuned adapter** on top of Mistral-7B.

The adapter path defaults to `data/models/lora_adapter/final_adapter`. If training produced a different structure, update `ADAPTER_PATH` below.

In [ ]:
import os

# Adapter paths for all 3 LoRA configs
LORA_CONFIGS = {
    "lora_r16": "/content/MLOps-group-10/data/models/lora_adapter",
    "lora_r8" : "/content/MLOps-group-10/data/models/lora_adapter_r8",
    "lora_r32": "/content/MLOps-group-10/data/models/lora_adapter_r32",
}

for approach, base_dir in LORA_CONFIGS.items():
    # Prefer final_adapter subdirectory if it exists
    final = os.path.join(base_dir, "final_adapter")
    adapter_path = final if os.path.isdir(final) else base_dir
    if not os.path.isdir(adapter_path):
        print(f"[SKIP] {approach}: adapter not found at {adapter_path}")
        continue
    print(f"\nRunning inference for {approach} — adapter: {adapter_path}")
    # Output: data/results/{approach}_fold1_test.csv
    !python3 scripts/run_inference_all.py --approach {approach} --adapter-path {adapter_path} --fold 1 --split test


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Display results for all approaches for fold 1 test split
results = {}
for approach in ["baseline", "rag", "lora_r16", "lora_r8", "lora_r32"]:
    path = f"/content/MLOps-group-10/data/results/{approach}_fold1_test.csv"
    if not os.path.exists(path):
        print(f"[WARN] Results not found for {approach}: {path}")
        continue
    df = pd.read_csv(path)
    results[approach] = df
    print(f"\n=== {approach.upper()} Results (fold 1, test split) ===")
    display(df.head())

# Optionally, show summary metrics if present
if results:
    print("\nSummary metrics (if available):")
    for approach, df in results.items():
        if "accuracy" in df.columns:
            print(f"{approach}: accuracy = {df['accuracy'].mean():.4f}")
        if "f1" in df.columns:
            print(f"{approach}: f1 = {df['f1'].mean():.4f}")

---
## Section 9: Full Evaluation

Run the complete evaluation pipeline comparing all approaches:

| Approach | Description |
|----------|-------------|
| Baseline | Direct Mistral-7B prompting |
| RAG | Mistral-7B + FAISS retrieval |
| LoRA | Fine-tuned Mistral-7B adapter |
| Altman Z-Score | Classical financial model (benchmark) |

Metrics computed: Accuracy, Precision, Recall, F1, AUC-ROC.

Outputs:
- `data/results/metrics_summary.csv`
- `data/results/roc_curves.png`

In [ ]:
# Run full evaluation pipeline — compares all four approaches
# This also generates Altman Z-Score predictions automatically
!python3 scripts/evaluate_all.py

In [ ]:
# Load and display the metrics summary table
import pandas as pd
from IPython.display import display
import os

metrics_path = "/content/MLOps-group-10/data/results/metrics_summary.csv"

if os.path.isfile(metrics_path):
    metrics_df = pd.read_csv(metrics_path)
    print("Evaluation Metrics Summary")
    print("=" * 60)

    # Style numeric columns
    numeric_cols = metrics_df.select_dtypes(include="number").columns.tolist()

    def highlight_best(s):
        """Highlight the best value in each numeric column green."""
        if s.name not in numeric_cols:
            return [""] * len(s)
        is_best = s == s.max()
        return ["background-color: #d4edda; font-weight: bold" if v else "" for v in is_best]

    styled = (
        metrics_df.style
        .apply(highlight_best)
        .format({col: "{:.4f}" for col in numeric_cols})
        .set_caption("Comparison of Credit Risk Detection Approaches")
        .set_table_styles([{
            "selector": "th",
            "props": [("background-color", "#343a40"), ("color", "white"), ("font-weight", "bold")]
        }])
    )
    display(styled)
else:
    print(f"Metrics summary not found at {metrics_path}. Check evaluate_all.py output above.")

In [ ]:
# Display the ROC curves image
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

roc_path = "/content/MLOps-group-10/data/results/roc_curves.png"

if os.path.isfile(roc_path):
    img = mpimg.imread(roc_path)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title("ROC Curves — All Approaches", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print(f"ROC curve image not found at {roc_path}.")

In [ ]:
# Show a per-company prediction comparison across all approaches
import pandas as pd
from IPython.display import display
import os

RESULTS_DIR = "/content/MLOps-group-10/data/results"
LABELS_PATH = "/content/MLOps-group-10/data/labels/company_labels.csv"

labels = pd.read_csv(LABELS_PATH)[["ticker", "label", "risk_category"]].rename(
    columns={"label": "true_label", "risk_category": "true_risk"}
)

comparison = labels.copy()

for approach in ["baseline", "rag", "lora_r8", "lora_r16", "lora_r32"]:
    pred_path = os.path.join(RESULTS_DIR, f"{approach}_predictions.csv")
    if os.path.isfile(pred_path):
        df = pd.read_csv(pred_path)[["ticker", "predicted_label", "predicted_score"]]
        df = df.rename(columns={
            "predicted_label": f"{approach}_pred",
            "predicted_score" : f"{approach}_score",
        })
        comparison = comparison.merge(df, on="ticker", how="left")

# Mark correct predictions
for approach in ["baseline", "rag", "lora_r8", "lora_r16", "lora_r32"]:
    col = f"{approach}_pred"
    if col in comparison.columns:
        comparison[f"{approach}_correct"] = (comparison[col] == comparison["true_label"])

print("Per-Company Prediction Comparison")
print("Green cell = correct prediction")

def highlight_correct(val):
    if val is True:
        return "background-color: #d4edda"
    elif val is False:
        return "background-color: #f8d7da"
    return ""

correct_cols = [c for c in comparison.columns if c.endswith("_correct")]
display(
    comparison.style
    .map(highlight_correct, subset=correct_cols)
    .set_caption("Prediction Comparison (green = correct, red = wrong)")
)


---
## Section 10: Download Results

Zip and download the results folder and the LoRA adapter from Colab to your local machine.

- `results.zip` — all prediction CSVs, metrics summary, ROC curves, loss curve
- `lora_adapter.zip` — the trained LoRA adapter weights (for re-use or deployment)

In [ ]:
import os
import shutil
from google.colab import files

RESULTS_DIR = "/content/MLOps-group-10/data/results"
ADAPTER_DIR = "/content/MLOps-group-10/data/models/lora_adapter"

# Zip the results directory
results_zip = "/content/results"
print("Zipping data/results/ ...")
shutil.make_archive(results_zip, "zip", RESULTS_DIR)
results_zip_path = results_zip + ".zip"
size_mb = os.path.getsize(results_zip_path) / 1e6
print(f"Created {results_zip_path} ({size_mb:.1f} MB)")

# List contents
print("\nContents:")
for f in sorted(os.listdir(RESULTS_DIR)):
    fpath = os.path.join(RESULTS_DIR, f)
    sz = os.path.getsize(fpath) / 1e3
    print(f"  {f}  ({sz:.1f} KB)")

# Download
print("\nDownloading results.zip ...")
files.download(results_zip_path)

In [ ]:
# Download all LoRA adapters as a single zip
import os
import shutil
from google.colab import files

ADAPTER_DIRS = {
    "lora_r16": "/content/MLOps-group-10/data/models/lora_adapter",
    "lora_r8" : "/content/MLOps-group-10/data/models/lora_adapter_r8",
    "lora_r32": "/content/MLOps-group-10/data/models/lora_adapter_r32",
}

# Bundle all available adapters into one archive
staging = "/content/lora_adapters_all"
os.makedirs(staging, exist_ok=True)

found = []
for name, src in ADAPTER_DIRS.items():
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(staging, name))
        found.append(name)
        print(f"  Included: {name}")
    else:
        print(f"  [SKIP] {name} — not found at {src}")

if found:
    zip_path = "/content/lora_adapters_all"
    print(f"\nZipping {len(found)} adapter(s)...")
    shutil.make_archive(zip_path, "zip", "/content", "lora_adapters_all")
    size_mb = os.path.getsize(zip_path + ".zip") / 1e6
    print(f"Created lora_adapters_all.zip ({size_mb:.1f} MB)")
    print("Downloading...")
    files.download(zip_path + ".zip")
else:
    print("No adapter directories found. Run Section 7 (LoRA Training) first.")


In [ ]:
# Final summary — print W&B links and key output paths
import os

print("=" * 60)
print("  PIPELINE COMPLETE")
print("=" * 60)

output_files = {
    "Baseline predictions" : "/content/MLOps-group-10/data/results/baseline_predictions.csv",
    "RAG predictions"      : "/content/MLOps-group-10/data/results/rag_predictions.csv",
    "LoRA r=8  predictions": "/content/MLOps-group-10/data/results/lora_r8_predictions.csv",
    "LoRA r=16 predictions": "/content/MLOps-group-10/data/results/lora_r16_predictions.csv",
    "LoRA r=32 predictions": "/content/MLOps-group-10/data/results/lora_r32_predictions.csv",
    "Altman Z-Score"       : "/content/MLOps-group-10/data/results/altman_zscore_predictions.csv",
    "Metrics summary"      : "/content/MLOps-group-10/data/results/metrics_summary.csv",
    "ROC curves"           : "/content/MLOps-group-10/data/results/roc_curves.png",
    "LoRA adapter r=16"    : "/content/MLOps-group-10/data/models/lora_adapter",
    "LoRA adapter r=8"     : "/content/MLOps-group-10/data/models/lora_adapter_r8",
    "LoRA adapter r=32"    : "/content/MLOps-group-10/data/models/lora_adapter_r32",
}

print("\nOutput files:")
for name, path in output_files.items():
    exists = os.path.isfile(path) or os.path.isdir(path)
    status = "OK" if exists else "MISSING"
    print(f"  [{status:7s}] {name}")

print("\nW&B Dashboard:")
print("  https://wandb.ai → project: credit-risk-detection")
print("  Runs: lora-r8, lora-sft, lora-r32 (training) | evaluation")
print("=" * 60)


In [ ]:
# Run RAG inference for all folds (1-5) on the test split
import subprocess

for fold in range(1, 6):
    print(f"Running RAG inference for fold {fold} (test split)...")
    result = subprocess.run([
        "python3", "scripts/run_inference_all.py",
        "--approach", "rag",
        "--fold", str(fold),
        "--split", "test"
    ], capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
